## **Os campeões brasileiros apresentam um perfil estatístico diferente das demais equipes?**

Investigar se os campeões brasileiros apresentam um perfil estatístico diferente das demais equipes, comparando indicadores de desempenho ao longo das temporadas analisadas.

In [ ]:
# Importação das bibliotecas necessárias para a análise.

import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Carregamento da base de dados dos jogos do Campeonato Brasileiro (2018 a 2023).

df = pd.read_csv('campeonato_brasileiro_tratado.csv')
df.head()

### **Escolha das colunas relevantes para a análise.**
Colunas com informações dos indicadores de perfil de jogo (chutes, chutes no alvo, posse de bola, passes, precisão de passes, faltas, cartões amarelos e vermelhos, escanteios e pontos).

A coluna "pontos" não se caracteriza commo indicador de perfil de jogo, mas será utilizada para a análise de desempenho do time.

A coluna "ano" servirá para a divisão das analíses por temporada.

In [ ]:
indicadores = [
    'chutes',
    'chutes_no_alvo',
    'posse_de_bola',
    'passes',
    'precisao_passes',
    'faltas',
    'cartao_amarelo',
    'cartao_vermelho',
    'escanteios'
]

### **Tratamento dos dados estatísticos**

Apesar de a base já estar previamente tratada, alguns indicadores estão armazenados em formatos não numéricos, especialmente valores percentuais. Nesta etapa, esses dados serão convertidos para permitir os cálculos estatísticos da análise.


In [ ]:
# Verificação dos tipos de dados das colunas selecionadas.

df[indicadores].dtypes

In [ ]:
# Alteração dos tipos de dados das colunas selecionadas para o tipo float e retirada do símbolo de porcentagem (%) das colunas 'posse_de_bola' e 'precisao_passes'.

df['posse_de_bola'] = df['posse_de_bola'].str.replace('%', '').astype(float)
df['precisao_passes'] = df['precisao_passes'].str.replace('%', '').astype(float)

df[indicadores].dtypes

### **Estatísticas por equipe e temporada**

Como a base apresenta as estatísticas de cada partida, os dados serão agrupados por equipe e temporada, calculando-se a média de cada indicador selecionado.

In [ ]:
## Separamos os dados por ano e clube, trabalhando somente com as estatísticas presentes em 'indicadores'.

media_estatisticas = (
    df.groupby(['ano', 'clube'])[indicadores]   # Separamos os dados por ano e clube, trabalhando somente com as estatísticas presente em indicadores
    .mean()                                     # Calculamos a média de cada estatística para cada clube em cada ano
    .round(2)                                   # Arredondamos os valores para duas casas decimais (para melhor visualização)
    .reset_index()                              # Resetamos o índice do DataFrame resultante
)

media_estatisticas.head(20)

### **Classificação final por temporada**

A classificação final de cada temporada será obtida a partir da pontuação acumulada pelas equipes, permitindo identificar o campeão de cada edição analisada.

A coluna 'pontos' representa os pontos conquistados em cada partida, onde cada vitória equivale a 3 pontos, empate 1 ponto e derrota 0 ponto

In [ ]:
# agrupa os registros por ano e clube e soma todos os pontos conquistados.

classificacao = (
    df.groupby(['ano', 'clube'])['pontos']
    .sum()
    .reset_index()
)

# organização da tabela pelo ano e pela pontuação.

classificacao = classificacao.sort_values(
    ['ano', 'pontos'],
    ascending=[True, False]
)

# cria uma nova coluna chamada posicao.

classificacao['posicao'] = (
    classificacao.groupby('ano')['pontos']
    .rank(method='first', ascending=False)
    .astype(int)
)

# Sem cálculos de média, apenas identificando a colocação final de cada time por ano e quantidade de pontos.

classificacao = classificacao[
    ['ano', 'posicao', 'clube', 'pontos']
]

classificacao.head(20)

In [ ]:
# Descobrir os campeões da cada ano, ou seja, os clubes que ficaram na primeira posição.

campeoes = classificacao.loc[
    classificacao.groupby("ano")["pontos"].idxmax()
]

campeoes

### **Integração das estatísticas com a classificação**


As médias estatísticas de cada equipe serão relacionadas à sua respectiva classificação final em cada temporada, permitindo comparar o perfil dos campeões com o das demais equipes.

In [ ]:
# Criamos um novo DataFrame chamado 'dados_analise'. Efetuamos o merge dos dois DF's existentes (classificacao e media_estatisticas) com base nas colunas 'ano' e 'clube', que são comuns a ambos.

dados_analise = pd.merge(
    classificacao,
    media_estatisticas,
    on=['ano', 'clube']
)

# Criaçãod e uma nova coluna chamada 'classificacao final' que classifica os clubes como 'CAMPEÃO' se estiverem na primeira posição, 'REBAIXADO' se estiverem nas posições 17 a 20, e vazio para os demais.
# Serve para observarmos se até um time rebaixado pode superar o campeão em determinado indicador. O futebol não é uma ciência exata!

dados_analise['classificacao final'] = dados_analise['posicao'].apply(lambda x: 'CAMPEÃO' if x == 1 else 'REBAIXADO' if x in [17, 18, 19, 20] else '')

dados_analise.head(20)

### **Comparação entre campeões e demais equipes**

Comparar o perfil estatístico médio dos campeões com o de todas as demais equipes, independentemente da posição final.

In [ ]:
# criação dos Dadaframes 'campeoes' e demais para separar os clubes campeões dos demais clubes.

campeoes = dados_analise[
    dados_analise['posicao'] == 1
]

demais = dados_analise[
    dados_analise['posicao'] != 1
]

In [ ]:
# Média das estatísticas dos campeões e dos demais clubes.

media_campeoes = campeoes[indicadores].mean().round(2)

media_demais = demais[indicadores].mean().round(2)

In [ ]:
# Comparação entre os dataframes de médias dos campeões e dos demais clubes

comparacao = pd.DataFrame({
    'Campeões': media_campeoes,
    'Demais equipes': media_demais
})

# inclusão da coluna diferença percentual entre os campeões e os outros clubes.

comparacao['Diferença (%)'] = (
    (comparacao['Campeões'] - comparacao['Demais equipes'])
    / comparacao['Demais equipes'] * 100
).round(2)

comparacao

### **Comparação por temporada**

Até agora, a gente olhou para todas as temporadas juntas e tirou uma média geral. Mas agora queremos dar um passo diferente:
Em cada temporada, como o campeão se saiu em relação às outras equipes daquele ano?

Dessa forma, evitamos que essa média geral esconda diferenças importantes de cada temporada. Exemplo: o Palmeiras de 2018 não tem as mesmas estatísticas do Palmeiras de 2023

In [ ]:
# Média das equipes (exceto o campeão) por ano, para cada indicador.

media_demais_ano = (
    demais.groupby('ano')[indicadores]
    .mean()
    .round(2)
    .reset_index()
)

media_demais_ano

In [ ]:
# Dados dos campeões por ano, para cada indicador.

campeoes_ano = campeoes[
    ['ano', 'clube'] + indicadores
].copy()

campeoes_ano

In [ ]:
# utilizei o comando 'suffixes' para diferenciar as colunas com as estatísticas do campeão de cada ano e os demais clubes.

comparacao_ano = pd.merge(
    campeoes_ano,
    media_demais_ano,
    on='ano',
    suffixes=('_campeao', '_demais')
)
comparacao_ano

### **Diferença por temporada**

Já temos o Dataframe de comparação, agora queremos calcular, para cada indicador, quanto o campeão ficou acima ou abaixo da média das demais equipes naquele ano.

In [ ]:
# Cada item do for a seguir representa a diferença (percentual) do campeão em relação às demais (por temporada)

for indicador in indicadores:
    comparacao_ano[f'{indicador}_dif'] = (
        (comparacao_ano[f'{indicador}_campeao'] -
         comparacao_ano[f'{indicador}_demais'])
        / comparacao_ano[f'{indicador}_demais'] * 100
    ).round(2)

In [ ]:
colunas_diferenca = [
    'ano',
    'clube'
] + [f'{indicador}_dif' for indicador in indicadores]

comparacao_ano[colunas_diferenca]

### **Visualização dos resultados**

Quanto os campeões diferem das demais equipes em cada indicador?

In [ ]:
import plotly.graph_objects as go

In [ ]:
indicadores_grafico = comparacao.index.tolist()
diferenca_geral = comparacao['Diferença (%)'].tolist()

In [ ]:
botoes = [
    dict(
        label='GERAL',
        method='update',
        args=[
            {
                'x': [diferenca_geral],
                'y': [indicadores_grafico],
                'text': [diferenca_geral]
            },
            {
                'title': 'Diferença percentual entre campeões e demais equipes'
            }
        ]
    )
]

In [ ]:
for _, linha in comparacao_ano.iterrows():

    ano = linha['ano']
    clube = linha['clube']

    diferencas = [
        linha[f'{indicador}_dif']
        for indicador in indicadores
    ]

    botoes.append(
        dict(
            label=str(ano),
            method='update',
            args=[
                {
                    'x': [diferencas],
                    'y': [indicadores],
                    'text': [diferencas]
                },
                {
                    'title': f'{clube} ({ano}) × demais equipes'
                }
            ]
        )
    )

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=diferenca_geral,
        y=indicadores_grafico,
        orientation='h',
        text=diferenca_geral,
        texttemplate='%{text:.2f}%',
        customdata=comparacao[['Campeões', 'Demais equipes']],
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Campeões: %{customdata[0]:.2f}<br>'
            'Demais equipes: %{customdata[1]:.2f}<br>'
            'Diferença: %{x:.2f}%'
            '<extra></extra>'
        )
    )
)

In [ ]:
fig.update_layout(
    updatemenus=[
        dict(
            buttons=botoes,
            direction='down',
            showactive=True,
            x=1,
            y=1.15
        )
    ],
    xaxis_title='Diferença (%)',
    yaxis_title='Indicador',
    title='Diferença percentual entre campeões e demais equipes'
)

fig.show()

In [ ]:
comparacao_ano[colunas_diferenca]

# **Qual time consegue produzir mais gols a cada 100 passes realizados?**

Investigar quais equipes apresentam maior **eficiência na produção de gols em relação ao volume de passes realizados**, utilizando como indicador a quantidade de gols marcados a cada 100 passes. A análise considera as temporadas disponíveis na base, permitindo comparar o desempenho das equipes tanto de forma geral quanto por temporada.

### **Escolha das colunas relevantes para a análise.**
Colunas com informações dos indicadores de perfil de jogo (passes, mandante_Placar e visitante_Placar).

As colunas "clube", "mandante", "visitante" e "pontos" não se caracterizam commo indicadores de perfil de jogo, mas serão utilizadas para a análise de desempenho dos times.

A coluna "ano" servirá para a divisão das analíses por temporada.

In [ ]:
indicadores_2 = [
    'ano',
    'clube',
    'mandante',
    'visitante',
    'mandante_Placar',
    'visitante_Placar',
    'passes'
]

In [ ]:
df_gols = df[indicadores_2].copy()

### **Construir a quantidade de gols marcados**

Como os placares estão separados entre mandante e visitante, precisamos determinar quantos gols pertencem ao clube representado em cada linha.

In [ ]:
# A fim de facilitar as nossas análises e cálculos, faremos a criação da coluna 'gols_marcados' para armazenar os gols dos mandantes assim como dos visitantes.

df_gols['gols_marcados'] = np.where(
    df_gols['clube'] == df_gols['mandante'],
    df_gols['mandante_Placar'],
    df_gols['visitante_Placar']
)

df_gols.head(10)

In [ ]:
df_gols.columns

### **Agrupar os dados por clube e temporada**

Efetuar o agrupamento dos dados das colunas selecionadas e separá-los por clube/temporada

In [ ]:
# Gols por clube e temporada

gols_por_temporada = (
    df_gols.groupby(['ano', 'clube'])['gols_marcados']
    .sum()
    .reset_index()
)

In [ ]:
# ordenando por campeonato

gols_por_temporada = gols_por_temporada.sort_values(
    ['ano', 'gols_marcados'],
    ascending=[True, False]
)

gols_por_temporada

In [ ]:
# Somatório geral por time

gols_geral = (
    df_gols.groupby('clube')['gols_marcados']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

gols_geral

### **Calcular a eficiência**

Através da criação da coluna 'gols_por_100_passes', nos permitirá comparar clubes que tiveram volumes diferentes de passes.

In [ ]:
# Agrupamos os dados por ano e clube e somamos todos os gols e também os passes de um determinado clube por emporada.

eficiencia = (
    df_gols.groupby(['ano', 'clube'])[
        ['gols_marcados', 'passes']
    ]
    .sum()
    .reset_index()
)

In [ ]:
# Dividimos o total de gols pelo total de passes e multiplicamos o resultado por 100, obtendo a quantidade de gols marcados a cada 100 passes realizados.

eficiencia['gols_por_100_passes'] = (
    eficiencia['gols_marcados']
    / eficiencia['passes']
    * 100
).round(3)

In [ ]:
#Ordenados pela eficiência (do maior para o menor) Portanto, o primeiro colocado será o time mais eficiente entre 2018 e 2023.

eficiencia = eficiencia.sort_values(
    'gols_por_100_passes',
    ascending=False
)

eficiencia

### **Criar o ranking geral**

Ordenar os clubes pela eficiência para descobrir quais times-temporada apresentaram os maiores valores entre 2018 e 2023.

In [ ]:
# criando a coluna 'ranking' no Dataframe 'eificencia' usando a eficiência de cada time.

eficiencia['ranking'] = (
    eficiencia['gols_por_100_passes']
    .rank(method='min', ascending=False)
    .astype(int)
)

In [ ]:
# Selecionando as colunas necessárias para visualizar o ranking geral de eficiência entre 2018 e 2023.

ranking_eficiencia = eficiencia[
    [
        'ranking',
        'ano',
        'clube',
        'gols_marcados',
        'passes',
        'gols_por_100_passes'
    ]
].sort_values('ranking')

ranking_eficiencia

### **Analisar por temporada**

Separar o resultado por ano para evitar que o ranking geral esconda diferenças entre temporadas.

In [ ]:
# primeiro separamos os clubes por ano e dentro de cada ano criamos o ranking usando 'gols_por_100_passes'. O maior valor recebe posição 1.

eficiencia['ranking_temporada'] = (
    eficiencia.groupby('ano')['gols_por_100_passes']
    .rank(method='min', ascending=False)
    .astype(int)
)

In [ ]:
# Selecionamos apenas as colunas importantes e organizamos primeiro pelo ano e depois pela posição no ranking.

ranking_temporada = eficiencia[
    [
        'ano',
        'ranking_temporada',
        'clube',
        'gols_marcados',
        'passes',
        'gols_por_100_passes'
    ]
].sort_values(
    ['ano', 'ranking_temporada']
)

ranking_temporada

In [ ]:
# Por ano

ranking_temporada[
    ranking_temporada['ano'] == 2018

]

### **Relacionar eficiência e desempenho**

Ser eficiente em gols por passes está associado a terminar melhor no campeonato?

In [ ]:
# Aqui iremos encontrar o mesmo ano e clube nos DataFrames e reunir as informações em 'eficiencia_desempenho'. Esse será o DataFrame final com todas as informações que precisamos para a análise.

eficiencia_desempenho = pd.merge(
    eficiencia,
    dados_analise[['ano', 'clube', 'posicao', 'pontos']],
    on=['ano', 'clube']
)

In [ ]:
# Criamos um ranking de eficiência ofensiva para cada temporada, considerando a quantidade de gols marcados a cada 100 passes..
# Detalhe: só comparamos tal eficiência entre os clubes que disputaram a mesma temproada.

eficiencia_desempenho['ranking_eficiencia'] = (
    eficiencia_desempenho
    .groupby('ano')['gols_por_100_passes']
    .rank(method='min', ascending=False)
    .astype(int)
)

In [ ]:
# Somente os campeões e mostra sua posição do ranking de eficiencia

campeoes_letalidade = (
    eficiencia_desempenho[
        eficiencia_desempenho['posicao'] == 1
    ][
        [
            'ano',
            'clube',
            'pontos',
            'gols_marcados',
            'gols_por_100_passes',
            'ranking_eficiencia'
        ]
    ]
    .sort_values('ano')
)

campeoes_letalidade

### **Construir a visualização**

In [ ]:
fig = go.Figure()

anos = sorted(eficiencia_desempenho['ano'].unique())

for ano in anos:

    dados_ano = (
        eficiencia_desempenho[
            eficiencia_desempenho['ano'] == ano
        ]
        .sort_values('gols_por_100_passes')
    )

    nomes_clubes = [
        f"<b>🏆 {clube}</b>" if posicao == 1 else clube
        for clube, posicao in zip(
            dados_ano['clube'],
            dados_ano['posicao']
        )
    ]

    fig.add_trace(
        go.Bar(
            x=dados_ano['gols_por_100_passes'],
            y=nomes_clubes,
            orientation='h',
            text=dados_ano['gols_por_100_passes'],
            texttemplate='%{text:.3f}',
            customdata=dados_ano[
                ['posicao', 'gols_marcados', 'passes']
            ],
            hovertemplate=(
                '<b>%{y}</b><br>'
                'Eficiência: %{x:.3f} gols/100 passes<br>'
                'Gols: %{customdata[1]}<br>'
                'Passes: %{customdata[2]}<br>'
                'Posição no campeonato: %{customdata[0]}º'
                '<extra></extra>'
            ),
            name=str(ano),
            visible=(ano == anos[0])
        )
    )

botoes = []

for i, ano in enumerate(anos):

    visibilidade = [False] * len(anos)
    visibilidade[i] = True

    botoes.append(
        dict(
            label=str(ano),
            method='update',
            args=[
                {'visible': visibilidade},
                {'title': f'Eficiência ofensiva por passes — {ano}'}
            ]
        )
    )

fig.update_layout(
    title=f'Eficiência ofensiva por passes — {anos[0]}',
    xaxis_title='Gols a cada 100 passes',
    yaxis_title='Clube',
    updatemenus=[
        dict(
            buttons=botoes,
            direction='down',
            showactive=True,
            x=1,
            y=1.12
        )
    ],
    height=650
)

fig.show()